# v2 · 01 · Entrenamiento QLoRA del adaptador v2: mitigación del sesgo D1 (Colab T4)

**TraceGit · ISIA-119 · Grupo 3.** Reentrena el adaptador de `google/gemma-4-E2B-it` (paso *map*: diff → mensaje de
commit) para mitigar el sesgo medido en el adaptador v1 (`eval/resultados/especializacion_test_v1_1_r2_2026-11-21.md`):

- v1 respondió en el idioma del commit en el 83,8 % de los ejemplos en inglés y solo en el 39,1 % de los ejemplos en
  español (brecha de 44,7 pp, IC 95 % de 11,5 a 70,2).
- v1 repitió «Add files via upload» en el 32 % del test, porque se entrenó con el dataset v1.

**Qué cambia frente al v1** (detalle en `notebooks/v2/README.md`):

| | v1 | v2 |
|---|---|---|
| Dataset | v1 @ 8b5d793 (3080 de train, con plantillas de GitHub) | **v1.1** @ 42eed23 (2903 de train, sin plantillas) |
| Idioma de salida | no se indica | **explícito en el prompt de sistema** («Escribe el mensaje de commit en español/inglés.») |
| Español | distribución natural | **sobremuestreo ×1,5** (1875 → 2813 ejemplos) |
| Épocas | 2 (el mejor checkpoint fue el del final de la 1.ª) | **1** |
| LoRA, lr, lote, semilla, MAX_LEN | r 16, alpha 32, lr 2e-4, lote 16, semilla 42, 1536 | iguales |

**El entrenamiento sobrevive a un corte de Colab:** cada 50 pasos el `Trainer` sube el último checkpoint a
`brenpc/tracegit-gemma4-e2b-lora-v2` (privado, subcarpeta `last-checkpoint`). Si la sesión se cae:
reconectar, **Run All**, y la celda 6 retoma desde ese checkpoint. Duración estimada: unas 1,7 h (240 pasos, a
partir de los 0,41 min por paso del v1; sin medir).

**Cómo ejecutarlo:** VS Code → *Select Kernel* → Colab → **GPU T4** → *Run All*. La celda 2 pide el token de
Hugging Face (con permiso de **escritura**) en un cuadro de texto: no pegarlo en el notebook ni en el chat.

La evaluación (D1 «después») va en `notebooks/v2/02_generacion_evaluacion_v2.ipynb`.

## 1. Configuración

In [ ]:
MODELO = dict(repo="google/gemma-4-E2B-it", revision="4adf15748de9e1a0097b6274cee9c15836c41c37")
DATASET = dict(repo="brenpc/tracegit-commits", revision="42eed230d333d04b7ad196032a6fe1d88524e323", carpeta="v1.1")
REPO_ADAPTADOR = "brenpc/tracegit-gemma4-e2b-lora-v2"
SALIDA = "/content/adaptador_v2"

# Mitigación D1 (notebooks/v2/README.md). El mismo texto se usa en la evaluación (02_generacion_evaluacion_v2).
FACTOR_ES = 1.5                           # cada ejemplo en español aparece 1,5 veces en promedio
INSTR_IDIOMA = {"es": "Escribe el mensaje de commit en español.",
                "en": "Escribe el mensaje de commit en inglés."}

# PLAN.md §4.3 (iguales al v1, salvo las épocas)
SEED = 42
MAX_LEN = 1536
EPOCAS = 1
BATCH, GRAD_ACCUM = 1, 16                 # lote efectivo 16
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
PASOS_GUARDADO = 50                       # guarda, evalúa y sube a Hub cada 50 pasos
N_VALIDACION = 120                        # subconjunto fijo de validación para la pérdida durante el entrenamiento

## 2. Dependencias y token de Hugging Face

Mismas versiones que la prueba de humo. Sin `datasets`: el JSONL se lee directo y se evita el conflicto de `pyarrow`.

In [ ]:
%pip install -q "transformers==5.23.0" "peft==0.21.3" "bitsandbytes==0.50.2" "accelerate==1.16.0"

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Token de Hugging Face (permiso de escritura): ").strip()
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Sesión de Hugging Face:", whoami()["name"])

## 3. Entorno

In [ ]:
import json, math, platform, random, time
import numpy as np
import torch, transformers, peft, bitsandbytes, accelerate

assert torch.cuda.is_available(), "No hay GPU: conectar el kernel a un servidor Colab con T4."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ENTORNO = {
    "gpu": torch.cuda.get_device_name(0),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": transformers.__version__, "peft": peft.__version__,
    "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__,
}
print(json.dumps(ENTORNO, indent=2))

## 4. Dataset v1.1 (revisión fijada), idioma explícito y sobremuestreo del español

Se comprueba que no quede ninguna plantilla de la web de GitHub (misma expresión que `dataset/v1_1/depurar.py`). El idioma de salida se agrega al final del prompt de sistema en train, validación y test. El sobremuestreo se aplica **solo al train**; la validación y el test conservan su distribución natural.

In [ ]:
import re
from collections import Counter
from huggingface_hub import hf_hub_download

RE_PLANTILLA_WEB = re.compile(r"^(?:Add files via upload|(?:Create|Update|Delete) \S*[./]\S*"
                              r"|Rename \S*[./]\S* to \S*[./]\S*)$")

def cargar(split):
    ruta = hf_hub_download(DATASET["repo"], f"{DATASET['carpeta']}/{split}.jsonl", repo_type="dataset",
                           revision=DATASET["revision"])
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

def con_idioma(ej):
    """Copia del ejemplo con el idioma de salida explícito al final del prompt de sistema."""
    msgs = [dict(m) for m in ej["messages"]]
    assert msgs[0]["role"] == "system", "se esperaba un prompt de sistema"
    msgs[0]["content"] = msgs[0]["content"].rstrip() + " " + INSTR_IDIOMA[ej["meta"]["idioma"]]
    return {**ej, "messages": msgs}

crudo = {s: [con_idioma(x) for x in cargar(s)] for s in ("train", "validation", "test")}
print({s: len(v) for s, v in crudo.items()})
plantillas = sum(bool(RE_PLANTILLA_WEB.match(x["messages"][-1]["content"].strip())) for x in crudo["train"])
assert plantillas == 0, f"{plantillas} plantillas de GitHub en el train: no es la v1.1"

rng_ds = random.Random(SEED)
es = [x for x in crudo["train"] if x["meta"]["idioma"] == "es"]
extra = rng_ds.sample(es, round(len(es) * (FACTOR_ES - 1)))
train_v2 = crudo["train"] + extra
rng_ds.shuffle(train_v2)
DIST = {"natural": dict(Counter(x["meta"]["idioma"] for x in crudo["train"])),
        "sobremuestreado": dict(Counter(x["meta"]["idioma"] for x in train_v2))}
print("train por idioma:", DIST, "· total", len(train_v2))
print("ejemplo de prompt de sistema:", train_v2[0]["messages"][0]["content"])

## 5. Modelo en 4 bits, tokenización y enmascarado

Igual que en la prueba de humo: NF4 con *double quant* y cómputo en fp16 (la T4 no tiene bf16); las torres de visión y audio no se cuantizan; la pérdida se calcula solo sobre la respuesta, que termina en `<turn|>`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODELO["repo"], revision=MODELO["revision"])
if tok.chat_template is None:
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODELO["repo"], revision=MODELO["revision"]).chat_template
FIN_TURNO = "<turn|>"

import os, signal, subprocess

def vram_libre():
    return torch.cuda.mem_get_info()[0] / 2**30

def liberar_gpu():
    """Cierra los kernels viejos del mismo servidor (p. ej. la prueba de humo) que retienen la GPU. No toca este."""
    yo = os.getpid()
    sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
    print("Procesos en la GPU (pid, MiB):", sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader").strip() or "-")
    pids = {int(l.split()[0]) for l in sh("ps -eo pid,cmd | grep -E 'ipykernel|colab_kernel_launcher' | grep -v grep").splitlines()}
    pids |= {int(p) for p in sh("nvidia-smi --query-compute-apps=pid --format=csv,noheader").split() if p.isdigit()}
    for pid in sorted(pids - {yo, os.getppid()}):
        try:
            print("  cerrando el proceso", pid, sh(f"ps -o cmd= -p {pid}").strip()[:80]); os.kill(pid, signal.SIGKILL)
        except (ProcessLookupError, PermissionError) as e:
            print("  no se pudo cerrar", pid, type(e).__name__)
    time.sleep(5)

if vram_libre() < 11:
    print(f"Solo {vram_libre():.1f} GB libres: hay otro kernel usando la GPU. Intentando liberarla...")
    liberar_gpu()
libre, total = (x / 2**30 for x in torch.cuda.mem_get_info())
print(f"VRAM libre: {libre:.1f} de {total:.1f} GB")
if libre < 11:
    raise RuntimeError("La GPU sigue ocupada por un proceso fuera de este servidor. Finalizar todas las sesiones de Colab "
                       "y conectar un servidor nuevo con T4.")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16,
                         llm_int8_skip_modules=["vision_tower", "audio_tower", "embed_vision", "embed_audio", "lm_head"])
t0 = time.time()
model = AutoModelForMultimodalLM.from_pretrained(MODELO["repo"], revision=MODELO["revision"],
                                                 quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
print(f"Modelo cargado en {time.time()-t0:.0f}s · VRAM {torch.cuda.memory_allocated()/2**30:.2f} GB")

def construir(ej):
    msgs = ej["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    completo = tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
    resp = completo[len(prompt):] if completo.startswith(prompt) else msgs[-1]["content"] + FIN_TURNO + "\n"
    ids_p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids_r = tok(resp, add_special_tokens=False)["input_ids"]
    ids = (ids_p + ids_r)[:MAX_LEN]
    return {"input_ids": ids, "labels": ([-100] * len(ids_p) + ids_r)[:MAX_LEN], "truncado": len(ids_p) + len(ids_r) > MAX_LEN}

entreno = [construir(x) for x in train_v2]
rng = random.Random(SEED)
validacion = [construir(x) for x in rng.sample(crudo["validation"], N_VALIDACION)]

fin = tok.decode([t for t in entreno[0]["labels"] if t != -100])
assert FIN_TURNO in fin, f"La respuesta no termina en {FIN_TURNO}"
lon = [len(x["input_ids"]) for x in entreno]
print(f"train {len(entreno)} · val {len(validacion)} · tokens media {np.mean(lon):.0f} p95 {np.percentile(lon,95):.0f} "
      f"máx {max(lon)} · truncados {sum(x['truncado'] for x in entreno)}")

## 6. LoRA y retoma desde Hugging Face Hub si hubo un corte

In [ ]:
from peft import LoraConfig, get_peft_model
from huggingface_hub import HfApi, snapshot_download

if hasattr(model, "peft_config"):
    raise RuntimeError("El modelo ya tiene LoRA (se re-ejecutó la celda). Reiniciar el kernel y ejecutar todo.")

model.activation_checkpointing_enable(activation_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
                  target_modules=r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)")
model = get_peft_model(model, lora)
for n, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()
mods = {n.split(".lora_")[0] for n, _ in model.named_parameters() if "lora_" in n}
assert mods and not any(("vision" in m) or ("audio" in m) for m in mods)
print("módulos LoRA:", len(mods), "· entrenables:", sum(p.numel() for p in model.parameters() if p.requires_grad))

api = HfApi()
api.create_repo(REPO_ADAPTADOR, private=True, exist_ok=True)
RETOMAR = None
if any(f.startswith("last-checkpoint/") for f in api.list_repo_files(REPO_ADAPTADOR)):
    local = snapshot_download(REPO_ADAPTADOR, allow_patterns=["last-checkpoint/*"], local_dir=SALIDA)
    RETOMAR = os.path.join(local, "last-checkpoint")
    estado = json.load(open(os.path.join(RETOMAR, "trainer_state.json")))
    print(f"Se retoma desde el paso {estado['global_step']} (checkpoint en Hub).")
else:
    print("Entrenamiento nuevo (no hay checkpoint en Hub).")

## 7. Entrenamiento

Pérdida y gradiente por paso. En fp16 el `GradScaler` puede saltarse algún paso con gradiente `inf` al inicio (visto en la prueba de humo): es normal. Se detiene solo si la **pérdida** es NaN o si el gradiente es `inf` durante 5 pasos seguidos.

In [ ]:
import inspect
from transformers import Trainer, TrainingArguments, TrainerCallback

acepta_tti = "token_type_ids" in inspect.signature(model.get_base_model().forward).parameters

def collate(lote):
    largo = max(len(x["input_ids"]) for x in lote)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.full((len(lote), largo), pad); lab = torch.full((len(lote), largo), -100)
    att = torch.zeros((len(lote), largo), dtype=torch.long)
    for i, x in enumerate(lote):
        n = len(x["input_ids"])
        ids[i, :n] = torch.tensor(x["input_ids"]); lab[i, :n] = torch.tensor(x["labels"]); att[i, :n] = 1
    out = {"input_ids": ids, "attention_mask": att, "labels": lab}
    if acepta_tti:
        out["token_type_ids"] = torch.zeros_like(ids)
    return out

class Vigia(TrainerCallback):
    def __init__(self): self.log, self.inf_seguidos = [], 0
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs:
            return
        if "eval_loss" in logs:
            self.log.append({"paso": state.global_step, "eval_loss": logs["eval_loss"]})
            print(f"  ── paso {state.global_step}: eval_loss {logs['eval_loss']:.4f}")
            return
        if "loss" not in logs:
            return
        loss, g = logs["loss"], logs.get("grad_norm")
        self.log.append({"paso": state.global_step, "loss": loss, "grad_norm": g, "lr": logs.get("learning_rate")})
        g_ok = g is None or math.isfinite(g)
        self.inf_seguidos = 0 if g_ok else self.inf_seguidos + 1
        if state.global_step % 10 == 0 or not g_ok:
            print(f"paso {state.global_step:>4}/{state.max_steps} | loss {loss:.4f} | grad_norm {g}")
        if not math.isfinite(loss) or self.inf_seguidos >= 5:
            print("FALLA: pérdida NaN o gradiente inf persistente. Se detiene."); control.should_training_stop = True

vigia = Vigia()
args = TrainingArguments(
    output_dir=SALIDA, num_train_epochs=EPOCAS, per_device_train_batch_size=BATCH, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=0.03,
    optim="paged_adamw_8bit", fp16=True, bf16=False, seed=SEED, logging_steps=1,
    eval_strategy="steps", eval_steps=PASOS_GUARDADO, save_strategy="steps", save_steps=PASOS_GUARDADO,
    save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    prediction_loss_only=True, push_to_hub=True, hub_model_id=REPO_ADAPTADOR, hub_private_repo=True,
    hub_strategy="checkpoint", report_to="none", remove_unused_columns=False, activation_checkpointing=False,
)
trainer = Trainer(model=model, args=args, train_dataset=entreno, eval_dataset=validacion,
                  data_collator=collate, callbacks=[vigia])

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
resultado = trainer.train(resume_from_checkpoint=RETOMAR)
HORAS = (time.time() - t0) / 3600
print(f"Fin: {HORAS:.2f} h en esta sesión · pico VRAM {torch.cuda.max_memory_allocated()/2**30:.2f} GB")
print(resultado)

## 8. Guardar y publicar el adaptador final con su *model card*

In [ ]:
trainer.save_model(SALIDA)            # con load_best_model_at_end es el de menor eval_loss
historial = trainer.state.log_history
evals = [(h["step"], round(h["eval_loss"], 4)) for h in historial if "eval_loss" in h]
mejor = trainer.state.best_metric

REGISTRO = {
    "modelo_base": f"{MODELO['repo']}@{MODELO['revision']}",
    "dataset": f"{DATASET['repo']}@{DATASET['revision']}:{DATASET['carpeta']}",
    "mitigacion_d1": {"instruccion_idioma": INSTR_IDIOMA, "factor_es": FACTOR_ES, "distribucion_train": DIST},
    "ejemplos_train": len(entreno), "ejemplos_val_usados": len(validacion), "epocas": EPOCAS,
    "pasos": trainer.state.global_step, "lote_efectivo": BATCH * GRAD_ACCUM, "lr": LR,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "modulos": len(mods)},
    "eval_loss_por_paso": evals, "mejor_eval_loss": mejor, "mejor_checkpoint": trainer.state.best_model_checkpoint,
    "horas_ultima_sesion": round(HORAS, 2), "pico_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "entorno": ENTORNO, "fecha": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()),
}
with open(os.path.join(SALIDA, "registro_entrenamiento.json"), "w", encoding="utf-8") as f:
    json.dump({**REGISTRO, "log": historial}, f, ensure_ascii=False, indent=2)

filas = "\n".join(f"| {p} | {l} |" for p, l in evals)
CARD = f"""---
base_model: {MODELO['repo']}
library_name: peft
license: apache-2.0
language: [es, en]
datasets: [{DATASET['repo']}]
tags: [lora, qlora, git, commits, code]
---

# TraceGit · adaptador QLoRA v2 para Gemma 4 E2B-it

Adaptador LoRA que especializa `{MODELO['repo']}` (revisión `{MODELO['revision'][:7]}`) en **describir cambios de
código**: dado un diff de git, escribe un mensaje de commit. Es el paso *map* de TraceGit (proyecto integrador
ISIA-119, Grupo 3). Código: https://github.com/brenpc1443/tracegit

**Uso previsto:** paso interno de un sistema que genera informes de proceso con evidencia citada.
**No** sirve para decidir si un código lo escribió una persona o una IA.

**Cómo pedirle el idioma:** al final del prompt de sistema se agrega «{INSTR_IDIOMA['es']}» o
«{INSTR_IDIOMA['en']}». Sin esa frase el adaptador se usa fuera de su condición de entrenamiento.

## Diferencias con el v1 (mitigación del sesgo D1)

El v1 respondía en el idioma del commit en el 83,8 % de los ejemplos en inglés y en el 39,1 % de los ejemplos en
español, y repetía «Add files via upload» en el 32 % del test. El v2 usa el dataset v1.1 (sin plantillas de la web de
GitHub), el idioma de salida explícito en el prompt, el español sobremuestreado ×{FACTOR_ES}
({DIST['natural']} → {DIST['sobremuestreado']}) y {EPOCAS} época. El resto de la configuración es igual.

## Entrenamiento

| | |
|---|---|
| Dataset | `{DATASET['repo']}` @ `{DATASET['revision'][:7]}`, carpeta `{DATASET['carpeta']}`: {len(entreno)} ejemplos de entrenamiento con el sobremuestreo (commits humanos anteriores al 30-11-2022) |
| Método | QLoRA: base en NF4 con *double quant*, cómputo fp16; pérdida solo sobre la respuesta |
| LoRA | r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}; q,k,v,o,gate,up,down del modelo de lenguaje ({len(mods)} módulos) |
| Optimización | paged_adamw_8bit, lr {LR}, coseno, warmup 3 %, lote efectivo {BATCH*GRAD_ACCUM}, {EPOCAS} épocas, semilla {SEED} |
| Hardware | {ENTORNO['gpu']} (Colab gratuito) |
| Selección | checkpoint de menor pérdida en validación ({N_VALIDACION} ejemplos fijos): **{mejor}** |

| Paso | eval_loss |
|---|---|
{filas}

## Limitaciones

- La métrica de pérdida no mide la fidelidad al diff: la evaluación con ROUGE-L, exactitud de identificadores y
  brecha por idioma está en `notebooks/v2/02_generacion_evaluacion_v2.ipynb` y `eval/metricas_especializacion.py`.
- La pérdida de validación del v2 no es comparable con la del v1: cambian el dataset y el prompt.
- Hereda los sesgos del modelo base y del dataset (62 % de mensajes en español, proyectos tipo curso).
"""
with open(os.path.join(SALIDA, "README.md"), "w", encoding="utf-8") as f:
    f.write(CARD)

from huggingface_hub import upload_folder
info = upload_folder(repo_id=REPO_ADAPTADOR, folder_path=SALIDA,
                     allow_patterns=["adapter_model.safetensors", "adapter_config.json", "README.md",
                                     "registro_entrenamiento.json", "tokenizer*", "chat_template*"],
                     commit_message=f"Adaptador v2 final (mejor eval_loss {mejor})")
print(">>> Copiar este SHA en ADAPTADORES['v2'] de 02_generacion_evaluacion_v2.ipynb:", info.oid)
print("Publicado:", f"https://huggingface.co/{REPO_ADAPTADOR}", "@", info.oid)
print(json.dumps(REGISTRO, indent=2, ensure_ascii=False))

## 9. Comprobación rápida: base frente a adaptador v2 (5 ejemplos de prueba)

Ambos reciben el prompt con el idioma explícito. Se genera de a un ejemplo, sin relleno (con relleno a la izquierda Gemma 4 E2B devuelve vacío). Las métricas van en `02_generacion_evaluacion_v2.ipynb`.

In [ ]:
model.eval(); model.config.use_cache = True

def generar(msgs, adaptador=True, max_new=64):
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        if adaptador:
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        else:
            with model.disable_adapter():
                out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
    return tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

for x in random.Random(1).sample(crudo["test"], 5):
    print(f"[{x['meta']['idioma']} · {x['meta']['lenguaje']}]")
    print("  HUMANO   :", x["messages"][-1]["content"][:150])
    print("  BASE     :", generar(x["messages"], adaptador=False)[:150].replace("\n", " "))
    print("  ADAPTADOR:", generar(x["messages"], adaptador=True)[:150].replace("\n", " "))